In [ ]:
# 구글 드라이브 공유 링크로 파일 다운로드
!pip install -U gdown
!pip install -qqq datasets # huggingface's lib.
!pip install -qqq transformers==4.48.3
!pip install -qqq accelerate==0.28.0
!pip install tensorboard
!pip install -U accelerate

In [ ]:
import gdown
import zipfile
import os

# 파일 ID 입력
file_id = "14Yc5eDlrEvx4wWp6gtSiJ52ne_noEl9u"

# 다운로드 받을 파일 이름 지정
output = "train.zip"

# 다운로드 수행
gdown.download(f"https://drive.google.com/uc?id={file_id}", output, quiet=True)

# 결과 파일 저장 경로
os.makedirs("release", exist_ok=True)

# 압축 해제할 경로
extract_dir = "data"
os.makedirs(extract_dir, exist_ok=True)

# 압축 풀기
with zipfile.ZipFile("train.zip", 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print("압축 해제 완료:", extract_dir)

In [ ]:
# Task 3 에서 검색할 Top-k 유사 이미지 개수
# Task 3 진행시만 사용
TOP_K = 5

In [ ]:
# 추가 패키지 설치 시 '버전 정보' 꼭 명시하여 설치:
# !pip install name_of_package==X.X.X

# 버전 명시 안해주시면 TA가 테스트할 때 버전 충돌이 자주 발생합니다.
# 불이익 받지 않도록 버전 잘 명시해주시기 바랍니다.

In [ ]:
# ===== 설정 =====
STUDENT_ID = "202502204"
CLIP_NAME = "openai/clip-vit-base-patch32"   # 검색용 공개 CLIP (학습 불필요)
BATCH_SIZE = 64
# TOP_K 은 위 템플릿 셀에서 정의됨.
# style/fruit 라벨은 test_labels.csv 에서 읽음(아래 셀) → 별도 업로드 모델 불필요.
print("config ready")

In [ ]:
# --- test 이미지 수집 (data/ 하위 자동 탐색, test 우선, id 숫자 정렬) ---
from glob import glob
EXTS = (".jpg", ".jpeg", ".png")

def find_image_dir(base="data"):
    cands = [r for r, _, fs in os.walk(base)
             if any(f.lower().endswith(EXTS) for f in fs)]
    for c in cands:
        if "test" in c.replace("\\", "/").lower():
            return c
    return cands[0] if cands else base

IMG_DIR = find_image_dir("data")
files = [p for p in glob(os.path.join(IMG_DIR, "*")) if p.lower().endswith(EXTS)]
def id_key(p):
    stem = os.path.splitext(os.path.basename(p))[0]
    return (0, int(stem)) if stem.isdigit() else (1, stem)
files = sorted(set(files), key=id_key)
ids = [os.path.basename(p) for p in files]
print("image dir:", IMG_DIR, "| images:", len(files))

In [ ]:
# --- CLIP 임베딩 (코사인용 L2 정규화) ---
import torch
from PIL import Image
from transformers import CLIPModel, CLIPProcessor

device = "cuda" if torch.cuda.is_available() else "cpu"
clip = CLIPModel.from_pretrained(CLIP_NAME).to(device).eval()
proc = CLIPProcessor.from_pretrained(CLIP_NAME)

@torch.no_grad()
def embed(paths):
    imgs = [Image.open(p).convert("RGB") for p in paths]
    batch = proc(images=imgs, return_tensors="pt").to(device)
    feat = clip.get_image_features(**batch)
    if not torch.is_tensor(feat):          # transformers 5.x는 객체 반환 → (B,512) 투영 임베딩
        feat = feat.pooler_output
    return torch.nn.functional.normalize(feat, dim=-1)

embs = []
for i in range(0, len(files), BATCH_SIZE):
    embs.append(embed(files[i:i + BATCH_SIZE]).cpu())
embs = torch.cat(embs)  # (N, D), normalized
print("embeddings:", tuple(embs.shape))

In [ ]:
# --- style/fruit 라벨을 test_labels.csv 에서 읽기 (분류기 사용 안 함) ---
import csv

def find_label_csv(base="data"):
    hits = glob(os.path.join(base, "**", "*.csv"), recursive=True)
    for h in hits:                       # 'test' 우선 (자가검증 땐 train_labels.csv)
        if "test" in os.path.basename(h).lower():
            return h
    return hits[0] if hits else None

CSV_PATH = find_label_csv("data")
assert CSV_PATH, "labels CSV(test_labels.csv)를 data/ 에서 찾지 못했습니다."
lab = {}
with open(CSV_PATH, newline="") as f:
    for row in csv.DictReader(f):
        lab[row["file_name"].strip()] = (int(row["style"]), int(row["fruit"]))
style_lab = [lab[i][0] for i in ids]
fruit_lab = [lab[i][1] for i in ids]
print("labels csv:", CSV_PATH, "| labeled:", len(style_lab))

In [ ]:
# --- Top-K 검색 (자기 자신 제외) & 출력 ---
os.makedirs("release", exist_ok=True)
out_path = f"release/{STUDENT_ID}.test.task3.txt"
N = embs.size(0)
embs_d = embs.to(device)

with open(out_path, "w") as f:
    for i in range(0, N, BATCH_SIZE):
        q = embs_d[i:i + BATCH_SIZE]
        sims = q @ embs_d.t()                       # (b, N) cosine
        for r in range(q.size(0)):
            gi = i + r
            sims[r, gi] = -1.0                       # exclude self
            topk = torch.topk(sims[r], TOP_K).indices.cpu().tolist()
            items = [f"({ids[j]},{style_lab[j]},{fruit_lab[j]})" for j in topk]
            f.write("[" + ", ".join(items) + "]\n")
print("wrote", out_path)

In [ ]:
with open(out_path) as f:
    lines = f.read().splitlines()
print("lines:", len(lines), "| TOP_K =", TOP_K)
for ln in lines[:3]:
    print(ln)